# Agent Flow Debug (Step-by-Step)

This notebook runs the agents **one-by-one** (no FastAPI, no LangGraph) so you can see each agent's inputs/outputs.

Prereqs:
- Put keys in `.env.local` (recommended): `OPENAI_API_KEY` and `TAVILY_API_KEY` (and/or `ANTHROPIC_API_KEY` depending on `config/agents.yaml`).
- Install deps: `pip install -r requirements.txt`

Notes:
- This is intentionally minimal: no looping, no retries, no special error handling.
- Each agent reads provider/model from `config/agents.yaml`.


In [1]:
import sys
from pathlib import Path
import json
from datetime import datetime

# Find repo root (directory containing both `src/` and `config/`).
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not ((repo_root / "src").exists() and (repo_root / "config").exists()):
    repo_root = repo_root.parent

if not ((repo_root / "src").exists() and (repo_root / "config").exists()):
    raise RuntimeError(f"Could not locate repo root from cwd={Path.cwd()}")

# Ensure repo root is on sys.path so `import src.*` works reliably.
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.env import load_env
load_env()

from src.state.schema import PipelineState
from src.agents.coordinator import CoordinatorAgent
from src.agents.research import ResearchAgent
from src.agents.analysis import AnalysisAgent
from src.agents.writing import WritingAgent
from src.agents.quality import QualityAgent


In [10]:
def refresh_agent(agent):
      # Re-read prompts.yaml into the already-created instance
      agent.system_prompt = agent._get_system_prompt()
      return agent

In [3]:



def state_summary(state: dict) -> dict:
    def _n(key: str) -> int:
        v = state.get(key)
        return len(v) if isinstance(v, list) else 0

    return {
        "run_id": state.get("run_id"),
        "topic": state.get("topic"),
        "current_agent": state.get("current_agent"),
        "pipeline_status": state.get("pipeline_status"),
        "quality_iteration": state.get("quality_iteration"),
        "quality_verdict": state.get("quality_verdict"),
        "quality_score": state.get("quality_score"),
        "counts": {
            "research_queries": _n("research_queries"),
            "raw_sources": _n("raw_sources"),
            "deduplicated_sources": _n("deduplicated_sources"),
            "clinical_claims": _n("clinical_claims"),
            "evidence_gaps": _n("evidence_gaps"),
            "citations": _n("citations"),
            "quality_issues": _n("quality_issues"),
        },
        "report_word_count": state.get("report_word_count"),
        "error_message": state.get("error_message"),
    }


def build_initial_state(
    *,
    topic: str,
    scope_instructions: str = "",
    target_audience: str = "clinical practitioners",
    report_format: str = "clinical_brief",
    max_quality_iterations: int = 3,
) -> PipelineState:
    return PipelineState(
        run_id="notebook-" + datetime.now().strftime("%Y%m%d-%H%M%S"),
        topic=topic,
        scope_instructions=scope_instructions,
        target_audience=target_audience,
        report_format=report_format,
        requested_at=datetime.now(),
        research_queries=[],
        scope_boundaries={"in_scope": [], "out_of_scope": []},
        priority_subtopics=[],
        raw_sources=[],
        deduplicated_sources=[],
        research_summary="",
        clinical_claims=[],
        evidence_gaps=[],
        contradictions=[],
        statistical_findings=[],
        analysis_narrative="",
        report_sections={},
        report_markdown="",
        citations=[],
        report_word_count=0,
        quality_issues=[],
        quality_verdict="pending",
        quality_score=0.0,
        revision_instructions="",
        quality_iteration=0,
        max_quality_iterations=max_quality_iterations,
        should_revise=False,
        current_agent="",
        agent_history=[],
        pipeline_status="pending",
        error_message=None,
    )


In [4]:
# Edit these, then re-run cells top-to-bottom.
TOPIC = "Effectiveness of telemedicine interventions for Type 2 diabetes management in adults"
SCOPE = "Focus on adults 18+, exclude pediatric cases. Prefer evidence from 2018-2026."
AUDIENCE = "clinical practitioners"
REPORT_FORMAT = "clinical_brief"  # or "full_report"
MAX_QUALITY_ITERATIONS = 3

state = build_initial_state(
    topic=TOPIC,
    scope_instructions=SCOPE,
    target_audience=AUDIENCE,
    report_format=REPORT_FORMAT,
    max_quality_iterations=MAX_QUALITY_ITERATIONS,
)

state_summary(state)


{'run_id': 'notebook-20260308-114053',
 'topic': 'Effectiveness of telemedicine interventions for Type 2 diabetes management in adults',
 'current_agent': '',
 'pipeline_status': 'pending',
 'quality_iteration': 0,
 'quality_verdict': 'pending',
 'quality_score': 0.0,
 'counts': {'research_queries': 0,
  'raw_sources': 0,
  'deduplicated_sources': 0,
  'clinical_claims': 0,
  'evidence_gaps': 0,
  'citations': 0,
  'quality_issues': 0},
 'report_word_count': 0,
 'error_message': None}

## 1) Coordinator Agent
Generates `research_queries`, `scope_boundaries`, and `priority_subtopics`.

In [5]:
coordinator = CoordinatorAgent()
coordinator._current_run_id = state.get("run_id")

state = await coordinator.execute(state)

print("Coordinator summary:")
print(json.dumps({
    "research_queries": state.get("research_queries"),
    "scope_boundaries": state.get("scope_boundaries"),
    "priority_subtopics": state.get("priority_subtopics"),
}, indent=2))
print("\nState:")
print(json.dumps(state_summary(state), indent=2))


Coordinator summary:
{
  "research_queries": [
    "telemedicine OR telehealth OR \"remote consultation\" AND \"Type 2 diabetes\" AND adults AND (glycemic control OR HbA1c) AND (randomized controlled trial OR RCT) AND (2018:2026[dp])",
    "\"Type 2 diabetes\" AND (telemedicine OR telehealth OR mHealth OR mobile health OR remote monitoring) AND (self-management OR disease management OR lifestyle intervention) AND (HbA1c OR blood glucose OR hypoglycemia) AND adults[MeSH] AND (2018:2026[dp])",
    "\"Type 2 diabetes mellitus\" AND (video consultation OR telephone follow-up OR digital health platform) AND (treatment adherence OR medication adherence OR appointment adherence) AND (clinical outcomes OR hospitalization OR emergency visits) AND (2018:2026[dp])",
    "\"Type 2 diabetes\" AND (telemonitoring OR remote patient monitoring) AND (quality of life OR patient satisfaction OR patient-reported outcomes) AND adults AND (systematic review OR meta-analysis) AND (2018:2026[dp])",
    "\"Typ

## 2) Research Agent
Runs Tavily searches for `research_queries`, deduplicates sources, and writes `research_summary`.

Requires `TAVILY_API_KEY` in environment.

In [6]:
research = ResearchAgent()
research._current_run_id = state.get("run_id")

state = await research.execute(state)

print("Research summary:")
print(state.get("research_summary"))

sources = state.get("deduplicated_sources", [])
print(f"\nDeduplicated sources: {len(sources)}")
for i, s in enumerate(sources[:5], 1):
    print(f"{i}. [{s.get('evidence_level')}] {s.get('title')} ({s.get('relevance_score')})")
    print(f"   {s.get('url')}")

print("\nState:")
print(json.dumps(state_summary(state), indent=2))


Research summary:
Found 19 relevant sources after deduplication:
- 5 Expert Opinion(s)
- 5 Randomized Controlled Trial(s)
- 9 Systematic Review(s)

Top sources by relevance:
1. Home Telemonitoring of Patients With Type 2 Diabetes: A Meta-Analysis and Systematic Review - PMC (relevance: 1.00)
2. The impact of telehealth remote patient monitoring on glycemic control in type 2 diabetes: a systematic review and meta-analysis of systematic reviews of randomised controlled trials - PMC (relevance: 1.00)
3. Evaluating the Impact of Telemedicine on Diabetes ... (relevance: 1.00)

Deduplicated sources: 19
1. [Systematic Review] Home Telemonitoring of Patients With Type 2 Diabetes: A Meta-Analysis and Systematic Review - PMC (0.99998236)
   https://pmc.ncbi.nlm.nih.gov/articles/PMC8914593
2. [Systematic Review] The impact of telehealth remote patient monitoring on glycemic control in type 2 diabetes: a systematic review and meta-analysis of systematic reviews of randomised controlled trials - PM

## 3) Analysis Agent
Extracts `clinical_claims`, `evidence_gaps`, `contradictions`, `statistical_findings`, and `analysis_narrative`.

In [7]:
analysis = AnalysisAgent()
analysis._current_run_id = state.get("run_id")

state = await analysis.execute(state)

print("Analysis narrative:\n")
print(state.get("analysis_narrative"))

claims = state.get("clinical_claims", [])
print(f"\nClinical claims: {len(claims)}")
for i, c in enumerate(claims[:5], 1):
    print(f"{i}. {c.get('claim')} ({c.get('evidence_level')})")
    if c.get("effect_size"):
        print(f"   effect_size: {c.get('effect_size')}")
    if c.get("p_value"):
        print(f"   p_value: {c.get('p_value')}")
    if c.get("source_urls"):
        print(f"   sources: {c.get('source_urls')}")

gaps = state.get("evidence_gaps", [])
print(f"\nEvidence gaps: {len(gaps)}")
for g in gaps[:10]:
    print(f"- {g}")

print("\nState:")
print(json.dumps(state_summary(state), indent=2))


Analysis narrative:

Across multiple systematic reviews and meta-analyses of randomized controlled trials, telemedicine interventions for adults with type 2 diabetes—particularly home telemonitoring and remote patient monitoring—consistently demonstrate modest improvements in glycemic control compared with usual care. Typical pooled HbA1c reductions fall in the range of about 0.3–0.6 percentage points, with most meta-analytic confidence intervals excluding no effect, although heterogeneity is substantial. Digital diabetes management technologies, including mobile apps, web portals, and CGM-based remote monitoring, also tend to improve self-management behaviors such as self-monitoring of blood glucose, medication adherence, and lifestyle behaviors. In older adults and in patient activation–focused programs, effect sizes on HbA1c are smaller (~0.2–0.4 percentage points) but still statistically significant, suggesting that telemedicine can provide incremental clinical benefit when layered

## 4) Writing Agent
Produces `report_sections`, `report_markdown`, `citations`, and `report_word_count`.

In [8]:
refresh = False

if refresh:
    analysis = refresh_agent(WritingAgent())
else: 
    writing = WritingAgent() 
writing._current_run_id = state.get("run_id")

state = await writing.execute(state)

print(f"Word count: {state.get('report_word_count')}")
print(f"Citations: {len(state.get('citations', []) or [])}")

print("\nReport preview:\n")
report = state.get("report_markdown", "") or ""
print(report[:2000])

print("\nState:")
print(json.dumps(state_summary(state), indent=2))


Word count: 3361
Citations: 10

Report preview:

## Executive Summary

Telemedicine interventions—including home telemonitoring, remote patient monitoring (RPM), mobile and web-based programs, and telehealth-enabled continuous glucose monitoring (CGM)—provide modest but consistent improvements in glycemic control and self-management for adults with type 2 diabetes (T2D) when added to usual care.

Across multiple systematic reviews and meta-analyses of randomized controlled trials, home telemonitoring and RPM reduce HbA1c by approximately 0.3–0.6 percentage points compared with usual care, with most pooled estimates statistically significant despite substantial heterogeneity.[1–4](https://pmc.ncbi.nlm.nih.gov/articles/PMC8914593)[https://pmc.ncbi.nlm.nih.gov/articles/PMC6019730/][https://pmc.ncbi.nlm.nih.gov/articles/PMC11260063/][https://pmc.ncbi.nlm.nih.gov/articles/PMC12168703) Patient activation–focused and older-adult–oriented interventions show smaller but still significant HbA1c 

## 5) Quality Agent
Scores the report and returns a verdict and issues.

This notebook runs quality once (no revision loop).

In [9]:
quality = QualityAgent()
quality._current_run_id = state.get("run_id")

state = await quality.execute(state)

print(f"Quality verdict: {state.get('quality_verdict')}")
print(f"Quality score: {state.get('quality_score')}")

issues = state.get("quality_issues", [])
print(f"\nIssues: {len(issues)}")
for i, issue in enumerate(issues[:20], 1):
    print(f"{i}. [{issue.get('severity')}] {issue.get('section')}: {issue.get('description')}")
    rec = issue.get('recommendation')
    if rec:
        print(f"   recommendation: {rec}")

print("\nRevision instructions:")
print(state.get("revision_instructions"))

print("\nState:")
print(json.dumps(state_summary(state), indent=2))


ValueError: Quality agent failed to parse response: LLM output appears truncated (agent=quality, model=gpt-5.1, max_tokens=2000). Increase this agent's max_tokens in config/agents.yaml. Parse error: Could not parse JSON from response: {
  "hard_failures": [],
  "dimension_scores": {
    "Clinical Accuracy": 21,
    "Completeness": 14,
    "Safety / Harms": 15,
    "Evidence Grading": 17,
    "Clarity / Audience Fit": 11
  },
  "qua